
# X-ray SED decomposition: AGN, LMXB, HMXB, hot gas

The X-CIGALE X-ray module (Yang et al. 2020) sums four physically distinct
emitters: the AGN corona (a cut-off power law normalized through the
α_OX–L_2500 relation), low- and high-mass X-ray binaries (LMXB ∝ M⋆, HMXB ∝ SFR;
Lehmer et al. 2016 metallicity/age scalings), and a hot interstellar-gas term
(∝ SFR). This reproduces Yang+2020 Figure 1 for a typical AGN host:
L_2–10 keV = 10⁴³ erg s⁻¹, M⋆ = 10¹¹ M⊙, SFR = 10 M⊙ yr⁻¹, T = 1 Gyr, Z = 0.02.

HMXB and LMXB are isolated from ``xray_xrb`` by driving the other population's
log-luminosity offset to −∞ (the ``det_hmxb`` / ``det_lmxb`` knobs).


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import warnings

import jax.numpy as jnp
import numpy as np
from matplotlib import pyplot as plt

from tengri.plot import setup_style
from tengri.xray import xray_agn_corona, xray_hotgas, xray_xrb

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

_KEV_AA = 12.398  # E[keV] = 12.398 / lambda[Å];  logE = log10(1.2398) - log10(lambda/nm)
_LOG_1P24 = np.log10(_KEV_AA / 10.0)

wave_aa = jnp.logspace(np.log10(0.01), np.log10(50.0), 600)
log_nm = np.log10(np.asarray(wave_aa) / 10.0)
nu_hz = 2.998e18 / np.asarray(wave_aa)
e_kev = _KEV_AA / np.asarray(wave_aa)

SFR, MSTAR, Z, AGE = 10.0, 1.0e11, 0.02, 1.0


def _l_2to10kev(l_nu):
    """L_2–10 keV = ∫ L_nu dν over the 2–10 keV band [erg/s]."""
    band = (e_kev >= 2.0) & (e_kev <= 10.0)
    order = np.argsort(nu_hz[band])
    return float(np.trapezoid(np.asarray(l_nu)[band][order], nu_hz[band][order]))


# AGN corona — rescale the model SED so L_2–10 keV = 1e43 erg/s (Yang Fig 1).
_BC_NU = 5.15 * 1.199e15
agn_raw = np.asarray(xray_agn_corona(wave_aa, l_2500_30deg_erg_hz=1.0e45 / _BC_NU))
agn = agn_raw * (1.0e43 / _l_2to10kev(agn_raw))

hmxb = np.asarray(
    xray_xrb(
        wave_aa,
        sfr=SFR,
        stellar_mass=MSTAR,
        metallicity_z=Z,
        stellar_age_gyr=AGE,
        log_L_lmxb_offset=-50.0,
    )
)
lmxb = np.asarray(
    xray_xrb(
        wave_aa,
        sfr=SFR,
        stellar_mass=MSTAR,
        metallicity_z=Z,
        stellar_age_gyr=AGE,
        log_L_hmxb_offset=-50.0,
    )
)
hotgas = np.asarray(xray_hotgas(wave_aa, sfr=SFR))

fig, ax = plt.subplots(figsize=(7.4, 5.0))
for sed, color, label in [
    (agn, "tab:blue", "AGN"),
    (lmxb, "tab:orange", "LMXB"),
    (hmxb, "tab:green", "HMXB"),
    (hotgas, "tab:red", "Hot gas"),
]:
    ax.plot(log_nm, np.log10(np.clip(sed, 1e-30, None)), color=color, lw=1.8, label=label)

ax.set(
    xlim=(-3.0, 0.7),
    ylim=(16.0, 26.0),
    xlabel=r"$\log\lambda$ (nm, rest-frame)",
    ylabel=r"$\log L_\nu$ (cgs)",
)

# Top axis: rest-frame energy. logE[keV] = log10(1.2398) - log10(lambda/nm).
ax_top = ax.secondary_xaxis(
    "top",
    functions=(lambda x: _LOG_1P24 - x, lambda lk: _LOG_1P24 - lk),
)
ax_top.set_xlabel(r"$\log E$ (keV)")

ax.legend(frameon=True, fontsize=10, loc="lower right", title="component")
fig.tight_layout()
plt.savefig("plot_xray_component_decomposition.png", dpi=150, bbox_inches="tight")